### Rag Pipeline - Data ingestion to Vector DB Pipeline

In [61]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [62]:
### Read all the pdf from directory
def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"\nProcessing:{len(pdf_files)} PDF file to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing:{pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
        except Exception as e:
            print(f"  ✗ Error: {e}")
            
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Processing:3 PDF file to process

Processing:Data_analyst.pdf
  ✓ Loaded 1 pages

Processing:Maaz    Amir (1).pdf
  ✓ Loaded 2 pages

Processing:MAAZ_Amir_AI_Resume.pdf
  ✓ Loaded 1 pages

Total documents loaded: 4


In [63]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-29T17:43:22+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-29T17:43:22+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Data_analyst.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Data_analyst.pdf', 'file_type': 'pdf'}, page_content='Maaz Amir\n♂phone +91-9934327856 ✉ amirmaaz16@gmail.com /linkedin LinkedIn /github GitHub\nSummary\nComputer Science graduate with internship experience in backend development and database-driven applications. Skilled in\nPython, relational databases, structured data processing, and problem-solving, with experience building REST APIs and\nvalidating application data. Interested in applying analytical thinking and data-driven approaches to business metrics,\noperational report

In [64]:
### Text splitting and get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    '''Split Documents into smaller chunks for better Rag performance'''
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [65]:
chunks = split_documents(all_pdf_documents)
chunks

Split 4 documents into 16 chunks

Example chunk:
Content: Maaz Amir
♂phone +91-9934327856 ✉ amirmaaz16@gmail.com /linkedin LinkedIn /github GitHub
Summary
Computer Science graduate with internship experience in backend development and database-driven applica...
Metadata: {'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-29T17:43:22+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-29T17:43:22+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Data_analyst.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Data_analyst.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-29T17:43:22+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-29T17:43:22+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Data_analyst.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Data_analyst.pdf', 'file_type': 'pdf'}, page_content='Maaz Amir\n♂phone +91-9934327856 ✉ amirmaaz16@gmail.com /linkedin LinkedIn /github GitHub\nSummary\nComputer Science graduate with internship experience in backend development and database-driven applications. Skilled in\nPython, relational databases, structured data processing, and problem-solving, with experience building REST APIs and\nvalidating application data. Interested in applying analytical thinking and data-driven approaches to business metrics,\noperational report

### Embedding and VectorStore DB

In [66]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import Any, Dict, List, Optional, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [67]:
class EmbeddingManager:
     """Handles document embedding generation using SentenceTransformer"""
     
     def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
     def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
     def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3705.41it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [68]:
# from importlib import metadata


class VectorStore:
    '''Manages document embedding in a ChromaDB Vector Store'''
    
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        '''
        Initialise the vector store
        
        Args:
            collection_name: Name of the ChromaDB Collection
            persist_directory: Directory to persist the vector store
        '''
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection=None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        '''
        Add documents and thier embedding to the vector store
        
        Args:
            documents: List of Langchain documents
            embedding: Corresponding embeddings for the documents
        '''
        
        if len(documents) != len(embeddings):
            raise ValueError('Number of documents must match number of embeddings')
        
        print(f"Adding {len(documents)} document to the vector store ...")
        
        # Prepare data for the ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list=[]
        
        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            # Generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document Content
            documents_text.append(doc.page_content)
            
            #Embeddings
            embeddings_list.append(embedding.tolist())
            
        # Add to collection
        try:
            if self.collection is None:
                raise RuntimeError("Vector store collection was not initialized")

            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
            

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 128


In [69]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-29T17:43:22+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-29T17:43:22+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Data_analyst.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Data_analyst.pdf', 'file_type': 'pdf'}, page_content='Maaz Amir\n♂phone +91-9934327856 ✉ amirmaaz16@gmail.com /linkedin LinkedIn /github GitHub\nSummary\nComputer Science graduate with internship experience in backend development and database-driven applications. Skilled in\nPython, relational databases, structured data processing, and problem-solving, with experience building REST APIs and\nvalidating application data. Interested in applying analytical thinking and data-driven approaches to business metrics,\noperational report

In [70]:
### Convert the Text to embeddings

texts = [doc.page_content for doc in chunks]

### Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

### Store everything in the vector database
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 16 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]

Generated embeddings with shape: (16, 384)
Adding 16 document to the vector store ...
Successfully added 16 documents to vector store
Total documents in collection: 144


### Retriever Pipeline from VectorStore

In [71]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: Optional[float] = None) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold. If None, return the top-k results.
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = (results.get("metadatas") or [[]])[0] or []
                distances = (results.get("distances") or [[]])[0] or []
                ids = (results.get("ids") or [[]])[0] or []
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # ChromaDB uses L2 distance by default. Convert it to a
                    # 0-1 similarity score while preserving the ranking.
                    similarity_score = 1 / (1 + distance)
                    
                    if score_threshold is None or similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [72]:
rag_retriever

In [73]:
rag_retriever.retrieve("Technical Skills")

Retrieving documents for query: 'Technical Skills'
Top K: 5, Score threshold: None
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 78.95it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_194fce0b_9',
  'content': 'Others: REST APIs, JWT, bcrypt.js, socket.io, Chart.js, Clerk, Shadcn\nAchievements\n• Solved 400+ problems on platform like Leetcode and GFG comprising of 10+ fundamental topics\n• Core Team Member, GDSC IIIT Bhopal — Organized and led multiple tech events and an alumni meetup, fostering\ncollaboration, networking, and knowledge sharing among 75+ participants\n• Attained maximum rating 1446 (2 stars) on CodeChef and solved 50+ problems\n• Selected as Teaching Assistant for Design and Analysis of Algorithms for top performance; mentored 70+ students',
  'metadata': {'author': '',
   'moddate': '2026-11-16T05:37:02+00:00',
   'title': '',
   'producer': 'pdfTeX-1.40.29',
   'creationdate': '2026-11-16T05:37:02+00:00',
   'total_pages': 2,
   'keywords': '',
   'creator': 'LaTeX with hyperref',
   'file_type': 'pdf',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2',
   'content_length': 

### RAG Pipeline- VectorDB To LLM Output Generation

In [74]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [75]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain.messages import HumanMessage, SystemMessage

In [76]:
class GroqLLM:
    def __init__(self, model_name: str = "qwen/qwen3.8-27b", api_key: Optional[str] = None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            api_key=SecretStr(self.api_key),
            model=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            content = response.content

            if isinstance(content, str):
                return content

            if isinstance(content, list):
                return "".join(
                    block
                    if isinstance(block, str)
                    else str(block.get("text", ""))
                    if isinstance(block, dict)
                    else str(block)
                    for block in content
                )

            return str(content)
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            response.content
            
            content = response.content
            
            if isinstance(content, str):
                return content
            
            if isinstance(content, list):
                return "".join(
                    block
                    if isinstance(block, str)
                                else str(block.get("text", ""))
                                if isinstance(block, dict)
                                else str(block)
                                for block in content
                            )
            
            return str(content)
        
        except Exception as e:
            return f"Error: {str(e)}"

In [77]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: qwen/qwen3.8-27b
Groq LLM initialized successfully!


In [78]:
rag_retriever.retrieve("Sumaary about education and projects done by Maaz")

Retrieving documents for query: 'Sumaary about education and projects done by Maaz'
Top K: 5, Score threshold: None
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.07it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_1788bfe6_15',
  'content': '• Guided students through 5+ hands-on projects, enhancing practical understanding of core concepts\nAchievements\n• Solved 400+ problems on platform like Leetcode and GFG comprising of 10+ fundamental topics\n• Secured 8th place among the top 10 teams in the GDSC IIIT Bhopal Ideathon contest\n• Core Team Member, GDSC IIIT Bhopal — Organized and led multiple tech events and an alumni meetup, fostering\ncollaboration, networking, and knowledge sharing among 75+ participants\n• Attained maximum rating 1446 (2 stars) on CodeChef and solved 50+ problems',
  'metadata': {'keywords': '',
   'content_length': 548,
   'source': '..\\data\\pdf\\MAAZ_Amir_AI_Resume.pdf',
   'page_label': '1',
   'total_pages': 1,
   'source_file': 'MAAZ_Amir_AI_Resume.pdf',
   'producer': 'pdfTeX-1.40.26',
   'file_type': 'pdf',
   'trapped': '/False',
   'page': 0,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.

### Integration Vectordb Context pipeline with LLM output

In [79]:
### Simple RAG pipeline with the Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from pydantic import SecretStr

load_dotenv()

### Initialise the GROQ LLM 
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError('GROQ API key is not found in your .env folder or not loaded properly')

llm = ChatGroq(
    api_key=SecretStr(groq_api_key),
    model="qwen/qwen3.8-27b",
    temperature=0.1,
    max_tokens=1024,
)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    # retrieve the context 
    results = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answer using the context
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content
    

In [80]:
answer = rag_simple("Professional summary about Maaz Amir.",rag_retriever,llm=llm)
answer

Retrieving documents for query: 'Professional summary about Maaz Amir.'
Top K: 3, Score threshold: None
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.53it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


'Maaz Amir is a Computer Science graduate with internship experience in backend development and database-driven applications. He is skilled in Python, relational databases, structured data processing, and problem-solving, with experience building REST APIs and validating application data. He is interested in applying analytical thinking and data-driven approaches to business metrics, operational reporting, and performance analysis.'

### Enhanced the RAG Pipeline with Features

In [81]:
# --- Enchanced RAG Pipeline with Features ---
def rag_advanced(query,retriever,llm,top_k=5,min_score=0.2,return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    
    results = retriever.retrieve(query,top_k=top_k,score_threshold=min_score)
    if not results:
        return {'answer':'No relevant context found',
                'sources':[],
                'confidence':0.0,
                'context':''}
        
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    
    sources = [{
        'source':doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke(prompt.format(context=context, query=query))
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

In [82]:
# Example usage:
result = rag_advanced("What is the overall achievements of Maaz?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
# if not result:
#     print( {
#         "answer": "No relevant context found",
#         "sources": [],
#         "confidence": 0.0,
#         "context": ""
#     })
print("Answer:", result['answer'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is the overall achievements of Maaz?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.92it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Based on the provided context, Maaz Amir's overall achievements include:

*   **Education:** Pursuing a B.Tech in Computer Science and Engineering at the Indian Institute of Information Technology, Bhopal, with a CGPA of 8.46/10.
*   **Professional Experience:** Completed an internship as a Backend Developer at Hike (Jul 2025 – Sep 2025), where he developed backend services and REST APIs using Node.js, Express.js, and TypeScript to support multi-scenario gameplay.
*   **Technical Skills:** Demonstrated proficiency in Python, relational databases, structured data processing, Redis, and Docker, with specific experience in building REST APIs and validating application data.
Confidence: 0.41991964838233026
Context Preview: Maaz Amir
♂phone +91-9934327856 ✉ amirmaaz16@gmail.com /linkedin LinkedIn /github GitHub
Summary
Computer Science graduate with internship experience in backend development and database-driven applications. Skilled in
Python, relational databases, structured data